In [ ]:
import os
import csv
import srt
import subprocess
import re
import shutil
from datetime import timedelta
import pytesseract
from pytesseract import Output
import cv2
import numpy as np

# --- CONFIGURATION ---
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

VIDEO_FILE = "commencement.mp4"
SRT_FILE = "commencement_shortened.srt"
FRAME_DIR = "frames"
CSV_FILE = "ocr_results.csv"

# Folders for sorted output
OCRED_DIR = os.path.join("ocr_results", "ocred")
NON_OCRED_DIR = os.path.join("ocr_results", "non-ocred")

os.makedirs(OCRED_DIR, exist_ok=True)
os.makedirs(NON_OCRED_DIR, exist_ok=True)
os.makedirs(FRAME_DIR, exist_ok=True)

TESSERACT_CONFIG = "--psm 6 --oem 3 -l eng -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'-. "

# Global memory to keep track of the last result
last_seen_text = None

def force_spaces(text):
    """Aggressive spacing fix for smashed words."""
    if not text: return ""
    # Space between lowercase and Uppercase
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
    # Space for joining words
    text = re.sub(r'([a-z])(of|the|and|for|in|to)\b', r'\1 \2', text, flags=re.IGNORECASE)
    return " ".join(text.split())

def format_timestamp(ts: timedelta):
    total_seconds = ts.total_seconds()
    h = int(total_seconds // 3600)
    m = int((total_seconds % 3600) // 60)
    s = total_seconds % 60
    return f"{h:02d}:{m:02d}:{s:06.3f}"

def extract_frame(timestamp: timedelta, label: str, index: int):
    ts_str = format_timestamp(timestamp)
    total_sec = timestamp.total_seconds()
    total_sec_str = f"{total_sec:.3f}".replace(".", "_")
    out_path = os.path.join(FRAME_DIR, f"{index}_s-{total_sec_str}_{label}.jpg")

    cmd = ["ffmpeg", "-ss", ts_str, "-i", VIDEO_FILE, "-frames:v", "1", "-y", out_path]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return out_path

def ocr_process(image_path):
    """Processes image and returns results without manual intervention."""
    global last_seen_text
    
    img = cv2.imread(image_path)
    if img is None: return "", 0

    h, w = img.shape[:2]
    # Crop coordinates (adjust if necessary)
    x0, y0 = 250, 520
    crop = img[y0:h, x0:w]

    # Perform OCR
    raw_text = pytesseract.image_to_string(crop, config=TESSERACT_CONFIG).strip()
    fixed_text = force_spaces(raw_text)

    # Get Confidence
    data = pytesseract.image_to_data(crop, config=TESSERACT_CONFIG, output_type=Output.DICT)
    valid_confs = [int(c) for c in data["conf"] if int(c) > 0]
    avg_conf = sum(valid_confs) / len(valid_confs) if valid_confs else 0.0

    # Print progress to console
    print(f"File: {os.path.basename(image_path)}")
    print(f"OCR Result: {fixed_text if fixed_text else '[NO TEXT FOUND]'}")
    print(f"Confidence: {avg_conf:.2f}")
    print("-" * 40)

    # Move files based on result
    filename = os.path.basename(image_path)
    if fixed_text:
        shutil.move(image_path, os.path.join(OCRED_DIR, filename))
    else:
        shutil.move(image_path, os.path.join(NON_OCRED_DIR, filename))

    return fixed_text, avg_conf

def run_pipeline():
    if not os.path.exists(SRT_FILE):
        print(f"Error: {SRT_FILE} not found!")
        return

    with open(SRT_FILE, "r", encoding="utf-8") as f:
        subs = list(srt.parse(f.read()))

    print(f"Starting automation for {len(subs)} subtitles...")

    with open(CSV_FILE, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["index", "timestamp", "label", "ocr_text", "confidence"])

        for sub in subs:
            mid = sub.start + (sub.end - sub.start) / 2
            
            # Process start, mid, and end frames for every subtitle
            frame_times = {
                "start": sub.start,
                "mid": mid,
                "end": sub.end
            }

            for label, ts in frame_times.items():
                path = extract_frame(ts, label, sub.index)
                text, conf = ocr_process(path)
                writer.writerow([sub.index, format_timestamp(ts), label, text, f"{conf:.2f}"])

    print("\nAutomation complete!")
    print(f"Results saved to: {CSV_FILE}")
    print(f"Images sorted into: {OCRED_DIR} and {NON_OCRED_DIR}")

if __name__ == "__main__":
    run_pipeline()

In [2]:
import csv
import json
import os

INPUT_CSV = "ocr_results.csv"
OUTPUT_JSON = "ocr_results_cleaned.json"

def csv_to_clean_json():
    if not os.path.exists(INPUT_CSV):
        print(f"Error: {INPUT_CSV} not found.")
        return

    cleaned = []
    last_seen_name = None

    with open(INPUT_CSV, mode="r", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)  # index, timestamp, label, ocr_text, confidence

        for row in reader:
            if len(row) < 4:
                continue

            index, timestamp, label, ocr_text, confidence = row
            name = ocr_text.strip()

            # Keep empty OCR rows (rare)
            if not name:
                cleaned.append({
                    "index": index,
                    "timestamp": timestamp,
                    "label": label,
                    "ocr_text": name,
                    "confidence": confidence
                })
                last_seen_name = ""
                continue

            # Only remove SEQUENTIAL duplicates
            if name != last_seen_name:
                cleaned.append({
                    "index": index,
                    "timestamp": timestamp,
                    "label": label,
                    "ocr_text": name,
                    "confidence": confidence
                })
                last_seen_name = name
            else:
                print(f"Skipping sequential duplicate: {name}")

    # Save JSON
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(cleaned, f, indent=2)

    print("-" * 40)
    print(f"Done! Cleaned entries: {len(cleaned)}")
    print(f"Saved to: {OUTPUT_JSON}")

if __name__ == "__main__":
    csv_to_clean_json()


Skipping sequential duplicate: Breyonna Brown
Skipping sequential duplicate: Breyonna Brown
Skipping sequential duplicate: Breyonna Brown
Skipping sequential duplicate: Breyonna Brown
Skipping sequential duplicate: Kayla Diane Howard Cum Laude
Skipping sequential duplicate: Kayla Diane Howard Cum Laude
Skipping sequential duplicate: Laura Esquivel
Skipping sequential duplicate: Laura Esquivel
Skipping sequential duplicate: Summer Denise Linton
Skipping sequential duplicate: Summer Denise Linton
Skipping sequential duplicate: Summer Denise Linton
Skipping sequential duplicate: Summer Denise Linton
Skipping sequential duplicate: Alyson Cochrane
Skipping sequential duplicate: Alyson Cochrane
Skipping sequential duplicate: Sean Dyami Andrews
Skipping sequential duplicate: Sean Dyami Andrews
Skipping sequential duplicate: Je Cory Clemons Robinson
Skipping sequential duplicate: Je Cory Clemons Robinson
Skipping sequential duplicate: Brett Bronistaw Duszak
Skipping sequential duplicate: Brett

In [3]:
import json

INPUT_JSON = "ocr_results_cleaned.json"
OUTPUT_JSON = "ocr_results_cleaned2.json"

def remove_timestamp_duplicates():
    with open(INPUT_JSON, "r", encoding="utf-8") as f:
        data = json.load(f)

    seen = set()
    cleaned = []

    for row in data:
        name = row["ocr_text"].strip()
        ts = row["timestamp"].strip()

        key = (ts, name)

        # Keep only the first time we see this timestamp+name pair
        if key not in seen:
            cleaned.append(row)
            seen.add(key)
        else:
            print(f"Removing timestamp duplicate: {name} @ {ts}")

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(cleaned, f, indent=2)

    print(f"Done. Original: {len(data)}, Cleaned: {len(cleaned)}")
    print(f"Saved to {OUTPUT_JSON}")

if __name__ == "__main__":
    remove_timestamp_duplicates()


Removing timestamp duplicate: Kimberly Denise Parker Magna Cum Laude @ 00:41:00.400
Removing timestamp duplicate: Faith Rosa Booker @ 00:41:06.200
Removing timestamp duplicate: Philip Azer Magna Cum Laude @ 00:43:02.960
Removing timestamp duplicate: Alexander J.Azpiazu Magna Cum Laude @ 00:43:54.800
Removing timestamp duplicate: Maisie Xuan Mai Tran @ 00:45:08.599
Removing timestamp duplicate: Chloe Zettler Robertson Magna Cum Laude @ 00:46:47.760
Removing timestamp duplicate: Conisha Rockeisha Scott @ 00:46:51.680
Removing timestamp duplicate: Ray'Neshia Rosalyn Michelle Smith @ 00:49:43.079
Removing timestamp duplicate: Jadyn Noel Howson Cum Laude @ 00:53:38.280
Removing timestamp duplicate: Jacob Ryan Smith @ 00:54:11.119
Removing timestamp duplicate: Georgia Sage Ginn @ 00:54:15.160
Removing timestamp duplicate: Carleigh Catherine Miley Magna Cum Laude @ 00:56:43.440
Removing timestamp duplicate: Sharonna Shanice Ray @ 00:58:39.559
Removing timestamp duplicate: Matty Layne Lenoir C